# Code

Running this notebook end-to-end will reproduce the solution. Step by step guide is also provided. You can skip some long running steps by executing corresponding cells of `Download.ipynb` to download artifacts.

In [ ]:
import sys
import os
import glob
import yaml

In [ ]:
!cat config.yaml

with open('config.yaml') as f:
    CONFIG = yaml.safe_load(f)
    
BASE_PATH = CONFIG['base_path']
CONFIG_PATH = os.path.join(BASE_PATH, 'config.yaml')
RAPIDS_ENV = os.path.join(BASE_PATH, CONFIG['rapids-pb-env'])
RAPIDS_GCN_ENV = os.path.join(BASE_PATH, CONFIG['rapids-gcn-env'])
PYTORCH_ENV = os.path.join(BASE_PATH, CONFIG['pytorch-env'])

# 1. Preparation

### 1.1. Setup envs

Create the following python envs:

* `pytorch-env` - env for all DL models (GCN training + inference, NN, embedding inference)
* `rapids-pb-env` - env for data preprocessing and training py-boost + logregs (RAPIDS 23.02, py-boost)
* `rapids-gcn-env` - env for GCN epoch evaluation and postprocessing (RAPIDS 26.06+, cudf for 64M-row TSV merges)

```bash
./create-rapids-pb-env.sh {BASE_PATH}
./create-pytorch-env.sh {BASE_PATH}
./create-rapids-gcn-env.sh {BASE_PATH}
```

In [ ]:
!./create-rapids-pb-env.sh {BASE_PATH}
!./create-pytorch-env.sh {BASE_PATH}
!./create-rapids-gcn-env.sh {BASE_PATH}

### 1.2. Get the input data

Here we describe what should be stored in the working dir to reproduce the results

Following data scheme was provided by Kaggle:

    ./Train - cafa train data
    ./Test (targets) - cafa test data
    ./sample_submission.tsv - cafa ssub
    ./IA.txt - cafa IA

    
Following are the solution code libraries, scipts, and notebooks used for training:

    ./protlib
    ./protnn
    ./nn_solution
    
And the installed envs

    ./pytorch-env
    ./rapids-pb-env
    ./rapids-gcn-env

### 1.3. Produce the helpers data

First, we made some preprocessing of the input data to store everything in format that is convinient to us to handle and manipulate. Here is the structure:

    ./helpers
        ./fasta - fasta files stored as feather
            ./train_seq.feather
            ./test_seq.feather
        ./real_targets - targets stored as n_proteins x n_terms parquet containing 0/1/NaN values
            ./biological_process
                ./part_0.parquet
                ...
                ./part_14.parquet
                ./prior.pkl - prior mean of each term (excluding NaN cells, like np.nanmean)
                ./nulls.pkl - NaN rate of each term
            ./cellular_component
            ./molecular_function

> **Mod 1:** Enable `--propagate True`. Original default was False, causing missing parent positives and inflated NaN rates in real_targets.
> GCN training comparison (SWA best):
> | Ontology |  propagate False |  propagate True | Gain |
> |------|------|------|------|
> | BP | 0.3275 | 0.4004 | **+0.073** |
> | MF | 0.6636 | 0.7048 | **+0.037** |
> | CC | 0.5983 | 0.6074 | **+0.009** |
>
> **Mod 2:** Added `drop_duplicates` + `trg[trg>0]=1.0` in create_helpers.py to prevent label values >1.

In [ ]:
%%time
# parse fasta files and save as feather
!{RAPIDS_ENV} protlib/scripts/parse_fasta.py \
    --config-path {CONFIG_PATH}

# convert targets to parquet and calculate priors
# use --propagate True to enable GO DAG upward closure before NaN generation
# optional: add --ontology <ns> to run each namespace in parallel (3x faster)
!{RAPIDS_ENV} protlib/scripts/create_helpers.py \
    --config-path {CONFIG_PATH} \
    --batch-size 10000 \
    --propagate True

### 1.4. Get external data

Datasets downloaded from outside and then processed. First step is downloading and parsing the datasets. After parsing, script will separate the datasets by the evidence codes. The most important split for us is kaggle/no-kaggle split. We refer `kaggle` as experimental codes, `no-kaggle` as electornic labeling, that will be used as features for the stacker models. Downloading takes quite a long time, while processing takes about 1 hour.

> **Note:** To avoid downloading the massive GOA database (~10h FTP), we provide a pre-packaged `./temporal/` directory with the required files already parsed.
> `prop_quickgo51.tsv` is an empty placeholder (header-only). The original solution used external QuickGO annotations for 37 proteins that we cannot reproduce. An empty file prevents `make_submission.py` from crashing without adding any external data.

The required structure after execution:

    ./temporal - extra data downloaded from http://ftp.ebi.ac.uk/pub/databases/GO/goa/old/UNIPROT/
    ./labels   - extracted and propagated labeling
        ./prop_test_leak_no_dup.tsv - leakage labeling
        ./prop_test_no_kaggle.tsv   - electronic labels test
        ./prop_train_no_kaggle.tsv  - electronic labels train
        
    ./cafa-terms-diff.tsv - reproduced difference between ML's dataset and our parsed labels (not used in CAFA6)
    ./prop_quickgo51.tsv  - reproduced MT's quickgo 37 proteins
    
    
Other files are temporary and not needed for future work

In [ ]:
# download external data from ebi.ac.uk
!{RAPIDS_ENV} protlib/scripts/downloads/dw_goant.py \
    --config-path {CONFIG_PATH}

# # parse the files
!{RAPIDS_ENV} protlib/scripts/parse_go_single.py \
    --file goa_uniprot_all.gaf.216.gz \
    --config-path {CONFIG_PATH}

!{RAPIDS_ENV} protlib/scripts/parse_go_single.py \
    --file goa_uniprot_all.gaf.214.gz \
    --config-path {CONFIG_PATH} \
    --output old214

The next step is propagation. Since ebi.ac datasets contains the labeling without propagation, we will apply the rules provided in organizer's repo to labeling more terms. We will do it only for `goa_uniprot_all.gaf.216.gz` datasets since it is the actual dataset at the active competition phase.

> **Note:** Our pre-packaged `./temporal/` data has already been propagated. This step can be skipped.

In [ ]:

folder = BASE_PATH + '/temporal'

for file in glob.glob(folder + '/labels/train*') + glob.glob(folder + '/labels/test*'):
    name = folder + '/labels/prop_' + file.split('/')[-1']

    !{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/prop_tsv.py \
        --path {file} \
        --graph {BASE_PATH}/Train/go-basic.obo \
        --output {name} \
        --device 0 \
        --batch_size 30000 \
        --batch_inner 5000

The last part is reproducing MT's datasets that are commonly used in all public kernels. We didn't use it directly, but we used `cafa-terms-diff` dataset, that represents the difference between our labeling obtained by parsing `goa_uniprot_all.gaf.216.gz` dataset and `all_dict.pkl` dataset given by MT. As he claims in the dicussion [here](https://www.kaggle.com/competitions/cafa-5-protein-function-prediction/discussion/404853#2329935) he used the same FTP source as we. But our source is more actual than the public. So the difference is actually the temporal. After analysis, we find out, that we are able to reproduce it as the difference between `goa_uniprot_all.gaf.216.gz` and `goa_uniprot_all.gaf.214.gz` sources. So, we just create `cafa-terms-diff` dataset by the given script. The only difference between the source in the kaggle script and used here is deduplication. We removed duplicated protein/terms pairs from the dataset, it has almost zero impact on the metric value (less than 1e-4)


In [ ]:
# create datasets
!{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/reproduce_mt.py \
    --path {BASE_PATH}/temporal \
    --graph {BASE_PATH}/Train/go-basic.obo

# # make propagation for quickgo51.tsv
!{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/prop_tsv.py \
    --path {BASE_PATH}/temporal/quickgo51.tsv \
    --graph {BASE_PATH}/Train/go-basic.obo \
    --output {BASE_PATH}/temporal/prop_quickgo51.tsv \
    --device 0 \
    --batch_size 30000 \
    --batch_inner 5000

### 1.5 Preparation step for neural networks

Produce some helpers to train NN model. Creates the following data:

    ./helpers/feats
        ./train_ids_cut43k.npy
        ./Y_31466_labels.npy
        ./Y_31466_sparse_float32.npz

In [ ]:
%%time

!{PYTORCH_ENV} {BASE_PATH}/nn_solution/prepare.py \
    --config-path {CONFIG_PATH}

# 2. Embeddings

> **Note:** There are pre-computed embeddings for all 3 types in `./embeds/`:
> - `./embeds/t5/` — ProtT5-XL (1024-dim)
> - `./embeds/esm_small/` — ESM2-650M (1280-dim)
> - `./embeds/struct/` — ESM-IF (Inverse Folding, 512-dim)
> 
> Skip sections 2.1–2.2 if using the pre-computed files.

The following is the coding method used by the cafa5-2nd team

In [ ]:
!mkdir embeds

### 2.1 T5 pretrained inference

    ./embeds
        ./t5
            ./train_ids.npy
            ./train_embeds.npy
            ./test_ids.npy
            ./test_embeds.npy

In [ ]:
%%time
!{PYTORCH_ENV} {BASE_PATH}/nn_solution/t5.py \
    --config-path {CONFIG_PATH} \
    --device 0

### 2.2 ESM pretrained inference

    ./embeds
        ./esm_small
            ./train_ids.npy
            ./train_embeds.npy
            ./test_ids.npy
            ./test_embeds.npy

In [ ]:
%%time
!{PYTORCH_ENV} {BASE_PATH}/nn_solution/esm2sm.py \
    --config-path {CONFIG_PATH} \
    --device 0

# 3. Base models

In [2]:
!mkdir models

mkdir: cannot create directory ‘models’: File exists


### 3.1. Train and inference py-boost models

GBDT models description:

1) Features: T5 + taxon, targets: multilabel

2) Features: T5 + taxon, targets: conditional

3) Features: T5 + ESM + taxon, targets: multilabel

4) Features: T5 + ESM + taxon, targets: conditional

Pipeline and hyperparameters are the same for all the models. Target is 4500 output: BP 3000, MF: 1000, CC: 500. All models could be ran in parallel to save a time. We used single V100 32GB and it requires about 15 hours to train 5 fold CV loop for each model type. 32GB GPU RAM is required, otherwise OOM will occur. Structure is:
    
    ./models
        ./pb_t54500_raw
            ./models_0.pkl
            ...
            ./models_4.pkl
            ./oof_pred.pkl
            ./test_pred.pkl
        ./pb_t54500_cond
            ...
        ./pb_t5esm4500_raw
            ...
        ./pb_t5esm4500_cond
            ...

In [ ]:
# 4 GBDT models (T5 + T5/ESM, raw + conditional). Run on separate GPUs for speed.
# Training time: ~2-3h per model on RTX 4080 32G.
for model_name in ['pb_t54500_raw', 'pb_t54500_cond', 'pb_t5esm4500_raw', 'pb_t5esm4500_cond']:

    print(f'Training {model_name}')

    !{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/train_pb.py \
        --config-path {CONFIG_PATH} \
        --model-name {model_name} \
        --device 0

# Optional: IF models (T5 + ESM-IF 512-dim). Add to list above if needed:
#   'pb_t5if4500_raw', 'pb_t5if4500_cond'

### 3.2. Train and inference logreg models

Logistic Regression models description:

1) Features: T5 + taxon, targets: multilabel

2) Features: T5 + taxon, targets: conditional


Pipeline and hyperparameters are the same for all the models. Target is 13500 output: BP 10000, MF: 2000, CC: 1500. All models could be ran in parallel to save a time. We used single V100 32GB and it requires about 10 hours for model 1 and 2 hours for model 2 to train 5 fold CV loop. 32GB GPU RAM is required, otherwise OOM will occur.

> **Mod 1:** Changed `alpha` from 1e-5 to 1e-7 in `train_lin.py`. At alpha=1e-5, NaN convergence failures (`delta_norm = nan`) were more frequent on sparse conditional terms. Lowering to 1e-7 reduced these failures.
> **Mod 2:** Added `--folds` option for parallel fold training + `merge_folds.py` to combine results.

Structure is:

    ./helpers
        ./folds_gkf.npy
    ./models
        ./lin_t5_raw
            ./models_0.pkl
            ...
            ./models_4.pkl
            ./oof_pred.pkl
            ./test_pred.pkl
        ./lin_t5_cond
            ...

In [ ]:
# lin_t5_raw supports parallel folds via --folds (e.g. --folds 0,1); merge with merge_folds.py after
for model_name in ['lin_t5_raw', 'lin_t5_cond']:

    print(f'Training {model_name}')

    !{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/train_lin.py \
        --config-path {CONFIG_PATH} \
        --model-name {model_name} \
        --device 0

### 3.3. Train and inference NN models

Structure is:

    ./models
        ./nn_serg
            ./model_0_0.pt
            ...
            ./model_11_4.pt
            ./pytorch-keras-etc-3-blend-cafa-metric-etc.pkl 

In [ ]:
# first, create train folds (the same as used for pb_t54500_cond model)
!{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/create_gkf.py \
    --config-path {CONFIG_PATH}

# train models (train_models.py: updated n_samples 142246→145382, test 141865→224309)
!{PYTORCH_ENV} {BASE_PATH}/nn_solution/train_models.py \
    --config-path {CONFIG_PATH} \
    --device 0

# inference models (inference_models.py: same data size updates)
!{PYTORCH_ENV} {BASE_PATH}/nn_solution/inference_models.py \
    --config-path {CONFIG_PATH} \
    --device 0

# reformat to use in stack
!{PYTORCH_ENV} {BASE_PATH}/nn_solution/make_pkl.py \
    --config-path {CONFIG_PATH}

# 4. Final model

### 4.1. Train GCN models

This step is training 3 independent stacking models for each ontology. Models are trained on single V100 GPU and it takes about 13 hours for BP, 4 hours for MF and 2 hours for CC. 32 GB GPU RAM is required to fit. Could be trained in parallel if 2 GPUs are avaliable - BP and MF/CC. Structure:

    ./models
        ./gcn
            ./bp
                ./checkpoint.pth
            ./mf
                ./checkpoint.pth
            ./cc
                ./checkpoint.pth
                

In [ ]:
%%time

for ont in ['bp', 'mf', 'cc']:
    !{PYTORCH_ENV} {BASE_PATH}/protnn/scripts/train_gcn.py \
        --config-path {CONFIG_PATH} \
        --ontology {ont} \
        --device 0

### 4.2. Inference GCN models and TTA

Inference and Test-Time-Augmentation. Structure:

    ./models
        ./gcn
            ./pred_tta_0.tsv
            ...
            ./pred_tta_3.tsv


In [ ]:
%%time

!{PYTORCH_ENV} {BASE_PATH}/protnn/scripts/predict_gcn.py \
    --config-path {CONFIG_PATH} \
    --device 0

### 4.3. Postprocessing and build submission file

Here we do the following:

1) Average TTA predictions
2) Perform min prop
3) Perform max prop
4) Average min/max prop steps, add external leakage data and make submission

Structure:

    ./models
        ./postproc
            ./pred.tsv     - avg TTA
            ./pred_min.tsv - min prop
            ./pred_max.tsv - max prop
            
    ./sub
        ./submission.tsv   - final results

In [ ]:
# since we have 4 TTA predictions, we need to aggregate all as an average
!{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/postproc/collect_ttas.py \
    --config-path {CONFIG_PATH} \
    --device 0

# create 0.3 * pred + 0.7 * max children propagation
!{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/postproc/step.py \
    --config-path {CONFIG_PATH} \
    --device 0 \
    --batch_size 30000 \
    --batch_inner 3000 \
    --lr 0.7 \
    --direction min

# create 0.3 * pred + 0.7 * min parents propagation
!{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/postproc/step.py \
    --config-path {CONFIG_PATH} \
    --device 0 \
    --batch_size 30000 \
    --batch_inner 3000 \
    --lr 0.7 \
    --direction max

# here we average min prop and max prop solutions, mix with cafa-terms-diff and quickgo51 datasets from 1.4
!{RAPIDS_ENV} {BASE_PATH}/protlib/scripts/postproc/make_submission.py \
    --config-path {CONFIG_PATH} \
    --device 0 \
    --max-rate 0.5

# Result

Result is stored in `./sub/submission.tsv`

In [ ]:
!head {BASE_PATH}/sub/submission.tsv